# Uitwerkingen

Deze uitwerkingen horen bij de herziene PGM2-oefentoets van 100 punten.
Er zijn ook andere geldige oplossingen die dezelfde contracten volgen.


## Opgave 1 *Extensies*

`check_extension` vergelijkt één naam en suffix en geeft een boolean terug.
Het lege suffix is het eerste basisgeval: daardoor passen ook twee lege strings
of twee gelijke strings. `allowed_file` beslist of één naam op minstens één
suffix past. `select_files` verwerkt de hele namenlijst en bewaart de passende
namen in volgorde.

Bij `spel.exe` roept `select_files` de functie `allowed_file` aan met die naam
en de twee extensies. Die vraagt `check_extension` eerst naar `.py` en dan naar
`.txt`. Beide resultaten zijn onwaar, dus de naam wordt niet opgenomen.
De onderste functie kent de namenlijst niet; de bovenste kent de afzonderlijke
lettervergelijkingen niet.

Voor hoofdletterongevoelige herkenning kan `check_extension` bij de aanroep
beide strings normaliseren met `lower()`. Daarna blijft hetzelfde recursieve
contract gelden. Beide aanroepende functies houden dezelfde interface en hoeven
niet te veranderen. Bestaande kleine-letteraanroepen behouden hun resultaat;
hoofdlettervarianten kunnen nu ook passen. Lege strings blijven correct.


In [ ]:
def check_extension(s, e):
    """Geef aan of de bestandsnaam eindigt op de gegeven extensie."""
    if e == "":
        return True
    if s == "" or s[-1] != e[-1]:
        return False
    return check_extension(s[:-1], e[:-1])


def allowed_file(filename, extensions):
    """Geef aan of een bestandsnaam op een toegestane extensie eindigt."""
    for extension in extensions:
        if check_extension(filename, extension):
            return True
    return False


def select_files(filenames, extensions):
    """Selecteer toegestane bestandsnamen met behoud van volgorde en duplicaten."""
    return [name for name in filenames if allowed_file(name, extensions)]


assert check_extension("", "") is True
assert check_extension("a", "a") is True
assert check_extension("", "a") is False
assert check_extension("a", "aa") is False
assert check_extension("program.exe", ".exe") is True
assert check_extension("tentamen.docx", ".exe") is False
assert check_extension("example.py", ".py") is True
assert check_extension("WERK.PY", ".py") is False
assert select_files(["werk.py", "spel.exe", "leeg.txt"], [".py", ".txt"]) == [
    "werk.py",
    "leeg.txt",
]
assert select_files(["a.py", "a.py"], [".py"]) == ["a.py", "a.py"]
assert select_files(["a"], []) == []
assert select_files(["a", ""], [""]) == ["a", ""]


## Opgave 2 *Even op vele manieren*

De comprehension filtert met een conditie. De recursie behandelt eerst de lege
lijst, en combineert daarna de eerste waarde met het resultaat voor de kortere
rest. De derde functie gebruikt het predicaat als parameter: de selectie hangt
daardoor af van de meegegeven functie.


In [ ]:
def only_even_lc(L):
    """Selecteer de even getallen met een list comprehension."""
    return [number for number in L if number % 2 == 0]


def only_even_rec(L):
    """Selecteer de even getallen recursief."""
    if L == []:
        return []
    rest = only_even_rec(L[1:])
    if L[0] % 2 == 0:
        return [L[0]] + rest
    return rest


def select_numbers(numbers, predicate):
    """Selecteer de getallen waarvoor het meegegeven predicaat waar is."""
    return [number for number in numbers if predicate(number)]


for function in [only_even_lc, only_even_rec]:
    assert function([]) == []
    numbers = [-2, -1, 0, 2, 2]
    assert function(numbers) == [-2, 0, 2, 2]
    assert numbers == [-2, -1, 0, 2, 2]
assert select_numbers([-2, 0, 3], lambda number: number > 0) == [3]
assert select_numbers([-2, 0, 3], lambda number: number % 2 == 0) == [-2, 0]
assert select_numbers([], lambda number: True) == []
calls = []
assert select_numbers([1, 2], lambda number: calls.append(number) or True) == [1, 2]
assert calls == [1, 2]


## Opgave 3 *Bonus*

Een toekenning aan een bestaande dictionarysleutel vervangt de vorige waarde.
Daarom verwerken we eerst alle registraties en geven we daarna pas de bonus.
Het setverschil neemt ook ingeschreven studenten zonder cijfer mee.


In [ ]:
records = [
    ["S01", 7.6, True],
    ["S02", 5.1, False],
    ["S03", 6.4, True],
    ["S01", 8.0, False],
]
participants = {"S01", "S02", "S03", "S04"}
students = {}
for student_number, grade, did_homework in records:
    students[student_number] = [grade, did_homework]
for student_number in students:
    if students[student_number][1]:
        students[student_number][0] += 0.5
overview = [[number, students[number][0]] for number in sorted(students)]
homework = {number for number in students if students[number][1]}
has_homework = "S03" in homework
missing = participants - homework
assert students == {"S01": [8.0, False], "S02": [5.1, False], "S03": [6.9, True]}
assert overview == [["S01", 8.0], ["S02", 5.1], ["S03", 6.9]]
assert homework == {"S03"}
assert has_homework is True
assert missing == {"S01", "S02", "S04"}


## Opgave 4 *Nim*

De zetmethode bewaakt de spelstaat, ook bij rechtstreeks gebruik. De besturing
probeert een beurt opnieuw na `ValueError`. Dezelfde aanroep `choose_move`
werkt voor beide spelers; de overschreven methode bepaalt het gedrag.
De interactieve cel wordt apart met vaste invoer gecontroleerd.


In [ ]:
from random import choice


class Nim:
    def __init__(self, number_of_sticks):
        """Bewaar het gegeven aantal lucifers als spelstaat."""
        self.sticks = number_of_sticks

    def player_takes(self, amount):
        """Voer een legale zet uit of gooi ValueError zonder de staat te wijzigen."""
        if type(amount) is not int or not 1 <= amount <= min(3, self.sticks):
            raise ValueError("Ongeldig aantal lucifers")
        self.sticks -= amount

    def game_over(self):
        """Geef aan of alle lucifers gepakt zijn."""
        return self.sticks == 0

    def __eq__(self, other):
        """Vergelijk het aantal lucifers van twee Nim-posities zonder mutatie."""
        return isinstance(other, Nim) and self.sticks == other.sticks


class Player:
    def choose_move(self, game):
        """Kies een zet zonder de spelstaat te wijzigen."""
        return int(input("Hoeveel lucifers pak je? "))


class StrategicPlayer(Player):
    def choose_move(self, game):
        """Kies een zet zonder de spelstaat te wijzigen."""
        remainder = game.sticks % 4
        if remainder:
            return remainder
        return choice(list(range(1, min(3, game.sticks) + 1)))


def play(game, players):
    """Speel om beurten en probeer dezelfde beurt opnieuw na een ongeldige zet."""
    turn = 0
    while not game.game_over():
        print("Aantal lucifers:", game.sticks)
        try:
            amount = players[turn].choose_move(game)
            game.player_takes(amount)
        except ValueError:
            print("Ongeldige zet")
            continue
        if game.game_over():
            print("Speler", turn + 1, "wint")
        else:
            turn = 1 - turn


for sticks in range(4):
    for amount in [-1, 0, 1, 2, 3, 4, 1.5, True]:
        game = Nim(sticks)
        if type(amount) is int and 1 <= amount <= min(3, sticks):
            assert game.player_takes(amount) is None
            assert game.sticks == sticks - amount
        else:
            try:
                game.player_takes(amount)
            except ValueError:
                assert game.sticks == sticks
            else:
                raise AssertionError("Ongeldige zet geaccepteerd")
assert Nim(0).game_over() is True
assert Nim(1).game_over() is False
left, right = Nim(2), Nim(2)
assert left == right
assert left != Nim(3)
assert left != 2
assert left.sticks == right.sticks == 2
computer = StrategicPlayer()
assert isinstance(computer, Player)
for sticks in range(1, 17):
    game = Nim(sticks)
    move = computer.choose_move(game)
    assert 1 <= move <= min(3, sticks)
    assert game.sticks == sticks
    if sticks % 4:
        assert move == sticks % 4


In [ ]:
# Player.choose_move vraagt input(); voer deze cel interactief uit.
play(Nim(16), [Player(), StrategicPlayer()])


## Opgave 5 *Alfabetisch*

Een woord met één letter voldoet. Bij langere woorden vergelijken we het
buurpaar en gaan we met een korter woord verder. `>` laat gelijke letters toe.


In [ ]:
def alfabet_word(s):
    """Geef aan of een niet-leeg kleinletterwoord alfabetisch geordend is."""
    if len(s) == 1:
        return True
    if s[0] > s[1]:
        return False
    return alfabet_word(s[1:])


assert "a" < "b"
assert alfabet_word("b") is True
assert alfabet_word("bel") is True
assert alfabet_word("adem") is True
assert alfabet_word("adders") is True
assert alfabet_word("aa") is True
assert alfabet_word("ab") is True
assert alfabet_word("ba") is False
assert alfabet_word("test") is False
assert alfabet_word("python") is False
assert alfabet_word("student") is False


## Opgave 6 *Moeilijke woorden*

Strategie: we berekenen aantallen voor de hele tekst en delen pas op het eind.
Zo telt elke complexe zin en elk complex woord eenmaal mee, ook wanneer een
complex woord in een complexe zin staat: het zijn verschillende elementen.

Structuur: één zelf ontworpen functie is hier voldoende. De comprehension maakt
een lijst van woordenlijsten per zin; die tussenstructuur dient zowel de
woordtelling als de zinstelling. Generator-expressies reduceren de aantallen.
Deze functionele keuze gebruikt geen verplichte extra hulpfuncties of klassen.
Een zelf ontworpen OO-oplossing met samenwerkende objecten is ook geldig als
de strategie, taakverdeling en hetzelfde scorecontract aantoonbaar zijn.


In [ ]:
def complexity_score(text):
    """Bereken de complexiteit van niet-lege, met punten afgesloten zinnen."""
    sentences = [sentence.split() for sentence in text.split(".")[:-1]]
    word_count = sum(len(words) for words in sentences)
    complex_words = sum(len(word) > 10 for words in sentences for word in words)
    complex_sentences = sum(len(words) > 15 for words in sentences)
    return 100 * (complex_words + complex_sentences) / (word_count + len(sentences))


In [ ]:
assert round(complexity_score("Dit is een korte zin."), 1) == 0.0
# Uitleg:
# Woorden (5): Dit(1) is(2) een(3) korte(4) zin(5)
# Zinnen (1): 1 zin van 5 woorden (< 15, dus niet complex)
# Score: 0 complexe elementen / 6 totale elementen = 0.0%

assert (
    round(
        complexity_score(
            "Dit is een gecompliceerde zin met veel verschillende woorden erin verstopt."
        ),
        1,
    )
    == 16.7
)
# Uitleg:
# Woorden (11): Dit(1) is(2) een(3) gecompliceerde(4) zin(5) met(6) veel(7)
#               verschillende(8) woorden(9) erin(10) verstopt(11)
# Zinnen (1): 1 zin van 11 woorden (< 15, dus niet complex)
# Complex: gecompliceerde(14 letters), verschillende(13 letters)
# Score: 2 complexe woorden / 12 totale elementen ≈ 16.7%

assert (
    round(complexity_score("De intelligentie van computerprogramma's neemt toe."), 1)
    == 28.6
)
# Uitleg:
# Woorden (6): De(1) intelligentie(2) van(3) computerprogramma's(4)
#              neemt(5) toe(6)
# Zinnen (1): 1 zin van 6 woorden (< 15, dus niet complex)
# Complex: intelligentie(13 letters), computerprogramma's(19 letters)
# Score: 2 complexe woorden / 7 totale elementen ≈ 28.6%

assert (
    round(
        complexity_score(
            "Programmeren is leuk. Dit is een zeer lange zin die meer dan vijftien woorden bevat en daarom als complex wordt beschouwd bij deze analyse. Nog een zin."
        ),
        1,
    )
    == 6.7
)
# Uitleg:
# Zin 1 (3): Programmeren(1) is(2) leuk(3)
# Zin 2 (21): Dit(1) is(2) een(3) zeer(4) lange(5) zin(6) die(7) meer(8) dan(9)
#             vijftien(10) woorden(11) bevat(12) en(13) daarom(14) als(15)
#             complex(16) wordt(17) beschouwd(18) bij(19) deze(20) analyse(21)
# Zin 3 (3): Nog(1) een(2) zin(3)
# Complex: programmeren(12 letters), en zin 2 (21 woorden > 15)
# Score: 2 complexe elementen / 30 totale elementen ≈ 6.7%
assert complexity_score("abcdefghij.") == 0.0
assert complexity_score("abcdefghijk.") == 50.0
assert complexity_score(" ".join(["a"] * 15) + ".") == 0.0
assert complexity_score(" ".join(["a"] * 16) + ".") == 100 / 17


De grenswaarden zijn strikt: tien tekens en vijftien woorden zijn nog niet
complex. De formule deelt door alle woorden plus alle zinnen; de invoerafspraak
zorgt dat deze noemer positief is. Rond alleen de controle-uitvoer af, zodat de
functie zelf haar volledige nauwkeurigheid behoudt.
